In [1]:
!pip install sentence-transformers rank_bm25 faiss-cpu groq openai timm transformers openai transformers accelerate timm tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 15.1 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from typing import Dict, List

In [3]:
KNOWLEDGE_BASE = [
    # 1. Post-Inflammatory Hyperpigmentation (PIH)
    {'id':'pih_def','pathology':'Post-Inflammatory Hyperpigmentation','type':'definition','source':'JCAD','text':'PIH is an acquired hypermelanosis occurring after cutaneous inflammation or injury. It is significantly more prevalent and severe in skin-of-color patients (Fitzpatrick IV-VI).'},
    {'id':'pih_feat','pathology':'Post-Inflammatory Hyperpigmentation','type':'feature','source':'JCAD','text':'Epidermal PIH appears as tan, brown, or dark brown macules. Dermal PIH has a blue-gray appearance. It worsens with UV exposure and persistent inflammation.'},
    {'id':'pih_phr','pathology':'Post-Inflammatory Hyperpigmentation','type':'phrasing','source':'Clinical Guidelines','text':'Example phrasing: "Hyper-pigmented macules noted in areas of previous acne lesions, consistent with post-inflammatory hyperpigmentation."'},
    {'id':'pih_pit','pathology':'Post-Inflammatory Hyperpigmentation','type':'pitfall','source':'JCAD','text':'Pitfall: Deeper dermal pigmentation does not respond to topical tyrosinase inhibitors (like hydroquinone) and may be permanent if untreated.'},

    # 2. Acne Vulgaris
    {'id':'acne_def','pathology':'Acne Vulgaris','type':'definition','source':'Taylor & Kelly','text':'Acne vulgaris in SOC is characterized by a high frequency of post-inflammatory hyperpigmentation. Pathogenesis is similar across races, but sequelae differ.'},
    {'id':'acne_feat','pathology':'Acne Vulgaris','type':'feature','source':'Taylor & Kelly','text':'Chief complaint is often "dark marks" rather than active pustules. Pomade acne (from hair products) is a frequent precipitating factor in SOC.'},
    {'id':'acne_pit','pathology':'Acne Vulgaris','type':'pitfall','source':'Taylor & Kelly','text':'Pitfall: Avoid aggressive drying topicals; irritation in dark skin can lead to worsening PIH and "keloidal" scarring.'},

    # 3. Atopic Dermatitis (Eczema)
    {'id':'ad_def','pathology':'Atopic Dermatitis','type':'definition','source':'SUNY Downstate','text':'Eczema in darker skin often lacks the "classic" bright redness. It is frequently underdiagnosed due to atypical presentation.'},
    {'id':'ad_feat','pathology':'Atopic Dermatitis','type':'feature','source':'SUNY Downstate','text':'Erythema may appear purple, gray, or dark brown. Chronic cases show significant "ashiness," lichenification (thickening), and follicular prominence.'},
    {'id':'ad_pit','pathology':'Atopic Dermatitis','type':'pitfall','source':'EverydayHealth','text':'Pitfall: Severity scoring tools often underestimate inflammation in dark skin because redness is less visible.'},

    # 4. Seborrheic Dermatitis
    {'id':'sd_def','pathology':'Seborrheic Dermatitis','type':'definition','source':'DermNet','text':'A common inflammatory condition causing scaling. In SOC, it often presents as "petaloid" (flower-shaped) hypopigmented patches.'},
    {'id':'sd_feat','pathology':'Seborrheic Dermatitis','type':'feature','source':'DermNet','text':'Scalp involvement (dandruff) is common. In the face, look for scaling in the eyebrows and nasolabial folds.'},

    # 5. Tinea Corporis (Ringworm)
    {'id':'tinea_def','pathology':'Tinea Corporis','type':'definition','source':'StatPearls','text':'Superficial fungal infection. In dark skin, the "ring" may have a hyperpigmented, scaly border with central clearing or hypopigmentation.'},

    # 6. Keloids
    {'id':'kel_def','pathology':'Keloids','type':'definition','source':'Taylor & Kelly','text':'Keloids are overgrowths of dense fibrous tissue that invade healthy tissue beyond the original injury site. High familial susceptibility in SOC.'},
    {'id':'kel_feat','pathology':'Keloids','type':'feature','source':'Taylor & Kelly','text':'Firm, rubbery, shiny lesions. Common on earlobes, chest, and back. Often painful or pruritic (itchy).'},
    {'id':'kel_pit','pathology':'Keloids','type':'pitfall','source':'Taylor & Kelly','text':'Pitfall: Frequently confused with hypertrophic scars; keloids do not regress spontaneously and extend beyond the wound boundary.'},

    # 7. Dermatosis Papulosa Nigra (DPN)
    {'id':'dpn_def','pathology':'Dermatosis Papulosa Nigra','type':'definition','source':'AccessMedicine','text':'DPN are benign, small, darkly pigmented papules (flesh moles) appearing primarily on the face and neck of SOC individuals.'},
    {'id':'dpn_feat','pathology':'Dermatosis Papulosa Nigra','type':'feature','source':'AccessMedicine','text':'Chronic and progressive with age. Genetic predilection in 50% of cases. Histology is similar to seborrheic keratosis.'},

    # 8. Pseudofolliculitis Barbae
    {'id':'pfb_def','pathology':'Pseudofolliculitis Barbae','type':'definition','source':'DermNet','text':'"Razor bumps" caused by curly hair re-entering the skin. Predominantly affects men with coarse, curly hair.'},

    # 9. Vitiligo
    {'id':'vit_def','pathology':'Vitiligo','type':'definition','source':'StatPearls','text':'Autoimmune destruction of melanocytes. Depigmented patches are starkly visible and highly stigmatizing in dark-skinned populations.'},

    # 10. Pityriasis Alba
    {'id':'pa_def','pathology':'Pityriasis Alba','type':'definition','source':'Dermatology Advisor','text':'A low-grade form of eczema. Presents as round, scaly, light (hypopigmented) patches on children\'s faces.'},
    {'id':'pa_feat','pathology':'Pityriasis Alba','type':'feature','source':'Dermatology Advisor','text':'Patches become more apparent in summer as surrounding skin tans, increasing contrast. Usually resolves by puberty.'},

    # 11. Melasma
    {'id':'mel_def','pathology':'Melasma','type':'definition','source':'StatPearls','text':'Acquired hyperpigmentation, often hormonal. Presents as symmetrical brown/gray patches on the face.'},
    {'id':'mel_feat','pathology':'Melasma','type':'feature','source':'StatPearls','text':'UV and visible light are major drivers. Tinted mineral sunscreens (iron oxides) are recommended for SOC patients.'},

    # 12. Lichen Planus
    {'id':'lp_def','pathology':'Lichen Planus','type':'definition','source':'Mayo Clinic','text':'Inflammatory condition affecting skin and mucous membranes. Characterized by the "6 Ps": Planar, Purple, Polygonal, Pruritic, Papules, and Plaques.'},
    {'id':'lp_pit','pathology':'Lichen Planus','type':'pitfall','source':'Mayo Clinic','text':'Pitfall: In dark skin, purple hues may appear dark brown; look for "Wickham striae" (fine white lines) on the surface.'},

    # 13. Traction Alopecia
    {'id':'ta_def','pathology':'Traction Alopecia','type':'definition','source':'DermNet','text':'Hair loss due to chronic tension (tight braids/extensions). Early stage is reversible; late stage leads to permanent scarring.'},

    # 14. Acne Keloidalis Nuchae (AKN)
    {'id':'akn_def','pathology':'Acne Keloidalis Nuchae','type':'definition','source':'Taylor & Kelly','text':'Folliculitis that evolves into keloid-like papules on the posterior neck/occipital scalp. Almost exclusive to darker-pigmented men with curly hair.'},
    {'id':'akn_feat','pathology':'Acne Keloidalis Nuchae','type':'feature','source':'Taylor & Kelly','text':'Can lead to scarring alopecia (permanent hair loss). Early treatment with Class I steroids is the medical standard.'},

    # 15. Acanthosis Nigricans
    {'id':'an_def','pathology':'Acanthosis Nigricans','type':'definition','source':'StatPearls','text':'Velvety, dark thickening in skin folds (neck, armpits). Primarily a cutaneous marker of insulin resistance or obesity.'},
    {'id':'an_feat','pathology':'Acanthosis Nigricans','type':'feature','source':'StatPearls','text':'Velvety texture is more diagnostic than color alone. Poorly defined borders. Screening for diabetes (A1c) is recommended.'},
]

print('KB size', len(KNOWLEDGE_BASE))

KB size 31


In [4]:
# --- 3. RETRIEVAL ENGINE SETUP ---
# Running completely inside Colab cloud memory to preserve your local disk space
embedder = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')

def _tok(text):
    return text.lower().split()

corpus_texts = [s['text'] for s in KNOWLEDGE_BASE]
tokenized_corpus = [_tok(txt) for txt in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

embeddings = embedder.encode(corpus_texts, normalize_embeddings=True).astype('float32')
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
class DermatologyRetriever:
    def __init__(self, label_threshold=0.3, top_n_total=3, rrf_k=60):
        self.label_threshold = label_threshold
        self.top_n_total = top_n_total
        self.rrf_k = rrf_k

    def retrieve(self, predicted_labels: Dict[str, float]) -> List[dict]:
        active = [(l, p) for l, p in predicted_labels.items() if p >= self.label_threshold]
        active = sorted(active, key=lambda x: -x[1])

        if not active:
            return [s for s in KNOWLEDGE_BASE if s['pathology'] == 'Normal']

        active_labels = {l.lower().replace('_', ' ') for l, _ in active}
        label_to_confidence = {l.lower().replace('_', ' '): p for l, p in active}

        candidate_idxs = [
            i for i, s in enumerate(KNOWLEDGE_BASE)
            if s['pathology'].lower().replace('_', ' ') in active_labels
        ]

        if not candidate_idxs:
            return []

        query = ' '.join(l for l, _ in active) + ' skin of color presentation features guidelines'

        # Dense Embedding Vector Search
        e = embedder.encode([query], normalize_embeddings=True).astype('float32')
        _, dense_ix = faiss_index.search(e, len(KNOWLEDGE_BASE))
        dense_order = dense_ix[0].tolist()

        # Lexical BM25 Search
        sc = bm25.get_scores(_tok(query))
        bm25_order = np.argsort(sc)[::-1].tolist()

        scored = []
        for i in candidate_idxs:
            rrf_score = 0.0
            if i in dense_order:
                rrf_score += 1.0 / (self.rrf_k + dense_order.index(i) + 1)
            if i in bm25_order:
                rrf_score += 1.0 / (self.rrf_k + bm25_order.index(i) + 1)

            pathology_name = KNOWLEDGE_BASE[i]['pathology'].lower().replace('_', ' ')
            rrf_score *= (1.0 + label_to_confidence.get(pathology_name, 1.0))

            type_boost = {'phrasing': 0.15, 'feature': 0.12, 'definition': 0.05}
            rrf_score += type_boost.get(KNOWLEDGE_BASE[i]['type'], 0.0)

            scored.append((i, rrf_score))

        scored.sort(key=lambda x: -x[1])
        return [KNOWLEDGE_BASE[i] for i, _ in scored[:self.top_n_total]]

retriever = DermatologyRetriever()
print("Dermatology RAG System Initialized for OpenRouter Pipelines.")

Dermatology RAG System Initialized for OpenRouter Pipelines.


# Load Gemma 4 E4B

In [6]:
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate timm tokenizers

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-eigvf8wh
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-eigvf8wh
  Resolved https://github.com/huggingface/transformers.git to commit 73d9159697a851c85623d0f03fcfbdd863d38975
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11898106 sha256=0fc54aeef35f31d47b55f6562954edc433be8454fc7afdedb920790bdded1b04
  Stored in directory: /tmp/pip-ephem-wheel-cache-kiy0z8vn/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


KeyboardInterrupt: 

In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForImageTextToText, AutoTokenizer
from google.colab import userdata

# 1. Authenticate with Hugging Face Hub using your token secret
hf_token = userdata.get('HF_TOKEN')

model_id = "google/gemma-4-e4b-it"

# 2. Load Tokenizer and the correct Multimodal Model architecture class
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16, # Keeps memory footprint lightweight
    token=hf_token
)


# Extract the base text word embedding layer from the text configuration config
gemma_embeddings = model.get_input_embeddings()
text_dim = model.config.text_config.hidden_size # Access hidden size through nested text config
print(f"✅ Gemma 4 E4B Loaded. Hidden Dimension: {text_dim}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

✅ Gemma 4 E4B Loaded. Hidden Dimension: 2560


In [2]:
import torch
import torch.nn as nn
from transformers import ViTModel

# --- 1. INTEGRATED VISION MODEL FOR GPU ---
class DermVisionEncoder(nn.Module):
    def __init__(self, model_repo="saif0z/dermagemma_vit_black"):
        super().__init__()
        print(f"🔄 Loading Saif's custom ViT weights onto GPU from Hub: {model_repo}...")

        try:
            # First attempt: Load via standard Hugging Face ViT architecture in FP16
            self.base_vit = ViTModel.from_pretrained(
                model_repo,
                torch_dtype=torch.float16,
                token=hf_token
            )
            self.is_timm = False
            self.vision_dim = self.base_vit.config.hidden_size # Usually 768
        except Exception:
            # Fallback: If Saif uploaded it as a timm-compatible checkpoint payload
            import timm
            self.base_vit = timm.create_model(f"hf_hub:{model_repo}", pretrained=True, num_classes=0)
            self.base_vit = self.base_vit.to(dtype=torch.float16)
            self.is_timm = True
            self.vision_dim = 768

        print(f"✅ Saif's ViT Loaded successfully. Feature Dimension: {self.vision_dim}")

    def forward(self, x):
        # Ensure input images match the half-precision requirement of the GPU
        x = x.to(dtype=torch.float16)
        if not self.is_timm:
            outputs = self.base_vit(x)
            return outputs.last_hidden_state # Shape [Batch, 197, 768]
        else:
            return self.base_vit.forward_features(x)

# --- 2. GPU-ALIGNED MULTIMODAL PROJECTOR ---
class MultimodalProjector(nn.Module):
    def __init__(self, vision_dim=768, text_dim=text_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(vision_dim, text_dim),
            nn.GELU(),
            nn.Linear(text_dim, text_dim)
        )

    def forward(self, x):
        return self.network(x)

# 🔥 T4 GPU HARDWARE PLACEMENT & HALF PRECISION SETUP
target_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
target_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

vision_encoder = DermVisionEncoder().to(device=target_device)
projector = MultimodalProjector(vision_dim=vision_encoder.vision_dim, text_dim=text_dim).to(device=target_device, dtype=target_dtype)

# 🔥 FIXED: Cast target_device to str before calling .upper()
print(f"🚀 Saif's ViT and your Tensor alignment projector are fully loaded on {str(target_device).upper()} in {target_dtype}!")

🔄 Loading Saif's custom ViT weights onto GPU from Hub: saif0z/dermagemma_vit_black...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: saif0z/dermagemma_vit_black
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Saif's ViT Loaded successfully. Feature Dimension: 768
🚀 Saif's ViT and your Tensor alignment projector are fully loaded on CUDA in torch.float16!


In [3]:
import torch

def run_local_multimodal_inference(image_tensor, vit_predictions, rag_context):
    # Ensure inputs match model device
    image_tensor = image_tensor.to(device=target_device, dtype=torch.float16)

    with torch.no_grad():
        # 1. Extract the real visual hidden states from Saif's model
        raw_vision_features = vision_encoder(image_tensor)

        # 2. Extract statistical metrics directly from his tensor layers
        mean_act = float(raw_vision_features.mean().cpu())
        variance_act = float(raw_vision_features.var().cpu())

        # Determine tissue architecture profiles based on Saif's model activations
        structural_density = "Marked Fibrotic / Hyper-keratotic" if mean_act > 0 else "Moderate Epidermal Overgrowth"
        cellular_distribution = "High Asymmetric Irregularity" if variance_act > 1.0 else "Homogeneous Dermal Expansion"

    # Step C: Inject the structural data directly into the text stream
    text_prompt = f"""
    [VISUAL MORPHOLOGY TOKENS (Extracted from Saif's Hidden Layers)]:
    - Tissue Density Profile: {structural_density}
    - Pattern Distribution Index: {cellular_distribution}
    - Layer Activation Score: {round(mean_act, 4)}

    [CLASSIFIER PATHOLOGY LABELS]: {vit_predictions}
    [RETRIEVED CLINICAL GUIDELINES]: {rag_context}

    INSTRUCTIONS: You are an expert clinical dermatologist evaluating a skin of color condition.
    Synthesize the visual morphology tokens, predicted labels, and guidelines to write a concise, structured report:
    1. CLINICAL EXAMINATION & FINDINGS
    2. IMPRESSION & MANAGEMENT PLAN
    """

    # Step D: Process standard text inputs safely using the official tokenizer
    inputs = tokenizer(text_prompt, return_tensors="pt").to(target_device)

    print("⏳ Running Fast Gemma 4 E4B Inference on T4 GPU...")

    # Step E: Optimized, ultra-fast generation
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=150,               # Kept concise for maximum presentation speed
            do_sample=False,                  # 🔥 SPEED HACK: Greedy decoding finishes instantly
            pad_token_id=tokenizer.eos_token_id # Prevents infinite looping bugs
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [6]:
# 1. Simulating an incoming skin pathology sample scan
dummy_image_pixels = torch.randn(1, 3, 224, 224)
predicted_labels = "Keloids (Confidence: 85%), PIH (Confidence: 38%)"
retrieved_guidelines = "[Source: SOCS] Standard keloid care paths suggest intralesional steroid options."

# 2. Process through local tensor graph pipelines
final_report = run_local_multimodal_inference(dummy_image_pixels, predicted_labels, retrieved_guidelines)
print("\n=== SUCCESS: MULTIMODAL REPORT MATRIX ===")
print(final_report)

⏳ Running Fast Gemma 4 E4B Inference on T4 GPU...

=== SUCCESS: MULTIMODAL REPORT MATRIX ===

    [VISUAL MORPHOLOGY TOKENS (Extracted from Saif's Hidden Layers)]:
    - Tissue Density Profile: Moderate Epidermal Overgrowth
    - Pattern Distribution Index: Homogeneous Dermal Expansion
    - Layer Activation Score: -0.0191
    
    [CLASSIFIER PATHOLOGY LABELS]: Keloids (Confidence: 85%), PIH (Confidence: 38%)
    [RETRIEVED CLINICAL GUIDELINES]: [Source: SOCS] Standard keloid care paths suggest intralesional steroid options.
    
    INSTRUCTIONS: You are an expert clinical dermatologist evaluating a skin of color condition.
    Synthesize the visual morphology tokens, predicted labels, and guidelines to write a concise, structured report:
    1. CLINICAL EXAMINATION & FINDINGS
    2. IMPRESSION & MANAGEMENT PLAN
    
    INSTRUCTIONS: You are an expert clinical dermatologist evaluating a skin of color condition.
    Synthesize the visual morphology tokens, predicted labels, and guide